# Construccion del dataset de elecciones

In [65]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import warnings
import unicodedata
import re

import numpy as np
import pandas as pd
import pyreadstat
import pycountry
from rapidfuzz import process, fuzz

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

# Detectar raiz del repo
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    for parent in ROOT.parents:
        if (parent / "data").exists():
            ROOT = parent
            break

paths = {
    "root": ROOT,
    "raw": ROOT / "data" / "raw",
    "interim": ROOT / "data" / "interim",
    "processed": ROOT / "data" / "processed",
    "manual": ROOT / "data" / "manual",
}
for p in paths.values():
    Path(p).mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

## 1. Funciones auxiliares

Funciones locales para limpiar nombres, construir fechas y resumir missings. 

In [66]:
def missingness_table(df: pd.DataFrame, top: int | None = None) -> pd.DataFrame:
    out = pd.DataFrame({
        "col": df.columns,
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "dtype": df.dtypes.astype(str),
    }).sort_values("missing_pct", ascending=False)
    if top is not None:
        out = out.head(top)
    return out


def normalize_country_name(name: str) -> str:
    if pd.isna(name):
        return ""
    s = str(name)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    s = s.replace("&", "and")
    for ch in [",", ".", "'", '"', "(", ")", "-"]:
        s = s.replace(ch, " ")
    s = " ".join(s.split())
    return s


def build_date_from_year_month(year: float | int, month: float | int) -> tuple[pd.Timestamp, str]:
    if pd.isna(year):
        return pd.NaT, "missing"
    y = int(year)
    if pd.isna(month) or int(month) < 1 or int(month) > 12:
        return pd.Timestamp(year=y, month=7, day=1), "year"
    return pd.Timestamp(year=y, month=int(month), day=1), "month"


def is_national_sub(val) -> bool:
    if pd.isna(val):
        return True
    # Si es numerico y negativo, lo tratamos como missing/nacional (codigos -99x)
    if isinstance(val, (int, float, np.integer, np.floating)):
        return val <= -990
    sval = str(val).strip().lower()
    return sval in {"-990", "-992", "-999", ""}


def map_cow_to_iso(cow_id, year, cow_df: pd.DataFrame) -> tuple[str | None, str]:
    if pd.isna(cow_id):
        return None, "missing_cow"
    try:
        cow_id = int(cow_id)
    except Exception:
        return None, "bad_cow"
    subset = cow_df[cow_df["cow_id"] == cow_id]
    if subset.empty:
        return None, "cow_not_found"
    if pd.notna(year):
        try:
            y = int(year)
            within = subset[(subset["valid_from"] <= y) & (subset["valid_until"] >= y)]
            if not within.empty:
                return within.iloc[0]["iso3"], "cow_year"
        except Exception:
            pass
    # Fallback: ultimo tramo conocido (si no hay valid_until, usar primera fila)
    subset = subset.copy()
    subset["valid_until"] = pd.to_numeric(subset["valid_until"], errors="coerce")
    if subset["valid_until"].notna().any():
        idx = subset["valid_until"].idxmax()
        return subset.loc[idx, "iso3"], "cow_latest"
    return subset.iloc[0]["iso3"], "cow_latest_no_year"


def build_country_name_map(
    raw_names: list[str],
    efw_name_map: dict[str, str],
    pycountry_map: dict[str, str],
    manual_overrides: dict[str, str | None],
    fuzzy_threshold: int = 90,
) -> pd.DataFrame:
    rows = []
    efw_choices = list(efw_name_map.keys())
    pyc_choices = list(pycountry_map.keys())

    for name in sorted(set(raw_names)):
        if pd.isna(name) or str(name).strip() == "":
            rows.append({
                "country_raw": name,
                "iso3": None,
                "match_method": "missing",
                "match_score": None,
                "matched_name": None,
            })
            continue
        if name in manual_overrides:
            rows.append({
                "country_raw": name,
                "iso3": manual_overrides[name],
                "match_method": "manual",
                "match_score": 100,
                "matched_name": name,
            })
            continue

        key = normalize_country_name(name)
        if key in efw_name_map:
            rows.append({
                "country_raw": name,
                "iso3": efw_name_map[key],
                "match_method": "efw_exact",
                "match_score": 100,
                "matched_name": key,
            })
            continue
        best = process.extractOne(key, efw_choices, scorer=fuzz.token_sort_ratio)
        if best and best[1] >= fuzzy_threshold:
            rows.append({
                "country_raw": name,
                "iso3": efw_name_map[best[0]],
                "match_method": "efw_fuzzy",
                "match_score": int(best[1]),
                "matched_name": best[0],
            })
            continue
        if key in pycountry_map:
            rows.append({
                "country_raw": name,
                "iso3": pycountry_map[key],
                "match_method": "pycountry_exact",
                "match_score": 100,
                "matched_name": key,
            })
            continue
        best = process.extractOne(key, pyc_choices, scorer=fuzz.token_sort_ratio)
        if best and best[1] >= fuzzy_threshold:
            rows.append({
                "country_raw": name,
                "iso3": pycountry_map[best[0]],
                "match_method": "pycountry_fuzzy",
                "match_score": int(best[1]),
                "matched_name": best[0],
            })
            continue

        rows.append({
            "country_raw": name,
            "iso3": None,
            "match_method": "unmatched",
            "match_score": None,
            "matched_name": None,
        })

    return pd.DataFrame(rows)

## 2. Inventario y trazabilidad de fuentes

Listado de archivos usados en este notebook (ubicacion y tamano). 

In [67]:
files = [
    paths["root"] / "efw.xlsx", # Solo para hacer evaluación de cobertura de países
    paths["root"] / "cow2iso.csv",
    paths["raw"] / "ned" / "presidential_elections_v2.dta",
    paths["raw"] / "ned" / "parliamentary_elections_v2.dta", 
    paths["raw"] / "clea" / "clea_lc_20251015.sav",
    paths["raw"] / "vparty" / "CPD_V-Party_CSV_v2" / "V-Dem-CPD-Party-V2.csv",
    paths["root"] / "data" / "external" / "partyfacts_core_parties.csv",
    paths["root"] / "data" / "external" / "partyfacts_external_parties.csv",
]

rows = []
for f in files:
    rows.append({
        "file": str(f),
        "exists": f.exists(),
        "size_mb": (f.stat().st_size / 1e6) if f.exists() else None,
    })

pd.DataFrame(rows)

,file,exists,size_mb
0,/Users/gabrielsaco/Documents/GitHub/economic-f...,True,5.419435
1,/Users/gabrielsaco/Documents/GitHub/economic-f...,True,0.018985
2,/Users/gabrielsaco/Documents/GitHub/economic-f...,True,9.019122
3,/Users/gabrielsaco/Documents/GitHub/economic-f...,True,51.936902
4,/Users/gabrielsaco/Documents/GitHub/economic-f...,True,540.756651
5,/Users/gabrielsaco/Documents/GitHub/economic-f...,True,14.290115
6,/Users/gabrielsaco/Documents/GitHub/economic-f...,True,1.528980
7,/Users/gabrielsaco/Documents/GitHub/economic-f...,True,9.204468


## 3. EFW: universo de paises y cobertura 2000+

In [68]:
efw_path = paths["root"] / "efw.xlsx"

ef = pd.read_excel(efw_path, sheet_name="EFW Panel Dataset")
ef = ef.rename(columns={
    "ISO_Code": "iso3",
    "Countries": "country",
    "Year": "year",
    "Summary": "efw_summary",
})

ef["iso3"] = ef["iso3"].astype(str).str.upper().str.strip()
ef["country"] = ef["country"].astype(str).str.strip()
ef["year"] = pd.to_numeric(ef["year"], errors="coerce")

efw_iso3 = sorted(ef["iso3"].dropna().unique().tolist())
ef_country_count = len(efw_iso3)
# Alias corto si se necesita en inspecciones
ef_iso3 = efw_iso3

# Cobertura 2000+
ef_2000 = ef[ef["year"] >= 2000].copy()
ef_2000_cov = ef_2000.groupby("year")["iso3"].nunique().reset_index(name="n_countries")

pd.DataFrame({
    "efw_unique_iso3": [ef_country_count],
    "efw_year_min": [int(ef["year"].min())],
    "efw_year_max": [int(ef["year"].max())],
})

,efw_unique_iso3,efw_year_min,efw_year_max
0,165,1970,2023


### 3.1 Mapas de nombres ISO3 (EFW + pycountry)

Creamos mapas de nombres para fallback y matching (reutilizados en NED y CLEA).

In [69]:
# Mapas de nombres EFW y pycountry (reutilizables)

ef_unique = ef[["iso3", "country"]].drop_duplicates()
# Usamos el nombre efw_name_map por consistencia en todo el notebook
efw_name_map = {normalize_country_name(n): iso for iso, n in zip(ef_unique["iso3"], ef_unique["country"])}

pyc_map = {}
for c in pycountry.countries:
    pyc_map[normalize_country_name(c.name)] = c.alpha_3
    if hasattr(c, "official_name"):
        pyc_map[normalize_country_name(c.official_name)] = c.alpha_3
    if hasattr(c, "common_name"):
        pyc_map[normalize_country_name(c.common_name)] = c.alpha_3

# Overrides manuales (casos ambiguos o no estandarizados)
manual_overrides = {
    "Korea": "KOR",
    "UK": "GBR",
    "Turkey": "TUR",
    "Kosovo": "XKX",
    "Macedonia": "MKD",
    "Cape Verde": "CPV",
    "Micronesia": "FSM",
    "Sint Maarten": "SXM",
    "Palestine": "PSE",
    "Macau": "MAC",
    "Saint Helena": "SHN"
}


## 4. COW -> ISO3 (para NED)

NED provee codigos COW. Usamos `cow2iso.csv` con vigencia por anio para mapear a ISO3, con fallback al tramo mas reciente si falta anio.

In [70]:
cow_path = paths["root"] / "cow2iso.csv"
cow = pd.read_csv(cow_path)

# Normalizamos tipos
cow["cow_id"] = pd.to_numeric(cow["cow_id"], errors="coerce")
cow["valid_from"] = pd.to_numeric(cow["valid_from"], errors="coerce")
cow["valid_until"] = pd.to_numeric(cow["valid_until"], errors="coerce")
cow["iso3"] = cow["iso3"].astype(str).str.upper().str.strip()

cow.head()

,cow_id,cow3,iso_id,iso2,iso3,valid_from,valid_until,cname,cname_full,comments,statenme
0,2.0,USA,841.0,US,USA,1962.0,1980.0,USA (before 1981),USA and Puerto Rico,Including Puerto Rico,United States of America
1,2.0,USA,842.0,US,USA,1981.0,NaN,USA,"USA, Puerto Rico and US Virgin Islands",Including Puerto Rico and US Virgin Islands,United States of America
2,20.0,CAN,124.0,CA,CAN,1962.0,NaN,Canada,Canada,NaN,Canada
3,31.0,BHM,44.0,BS,BHS,1962.0,NaN,Bahamas,Bahamas,NaN,Bahamas
4,40.0,CUB,192.0,CU,CUB,1962.0,NaN,Cuba,Cuba,NaN,Cuba


## 5. NED presidencial: top-2 y margenes

Regla: si existe segunda vuelta (>=2 candidatos con `vote_share2`), usamos la segunda vuelta. Si no, usamos primera vuelta. Esto produce un margen comparable para RD.

In [71]:
def extract_top2_presidential(df: pd.DataFrame) -> pd.DataFrame:
    cand_nums = sorted({int(re.findall(r"candidate_(\d+)", c)[0]) for c in df.columns if c.startswith("candidate_")})
    parts = []
    for i in cand_nums:
        cand_col = f"candidate_{i}"
        party_col = f"party_{i}"
        vs1_col = f"vote_share1_{i}"
        vs2_col = f"vote_share2_{i}"
        if not all(col in df.columns for col in [cand_col, party_col, vs1_col, vs2_col]):
            continue
        tmp = df[["source_election_id", cand_col, party_col, vs1_col, vs2_col]].copy()
        tmp.columns = ["source_election_id", "candidate", "party", "vote_share1", "vote_share2"]
        parts.append(tmp)

    long = pd.concat(parts, ignore_index=True)
    long["round2_available"] = long.groupby("source_election_id")["vote_share2"].transform(lambda s: s.notna().sum() >= 2)
    long["share"] = np.where(long["round2_available"], long["vote_share2"], long["vote_share1"])
    long = long[long["share"].notna()].copy()

    long = long.sort_values(["source_election_id", "share"], ascending=[True, False])
    long["rank"] = long.groupby("source_election_id").cumcount() + 1
    top2 = long[long["rank"] <= 2].copy()

    wide = top2.pivot(index="source_election_id", columns="rank", values=["candidate", "party", "share"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    round_used = long.groupby("source_election_id")["round2_available"].max().rename("round2_used")
    out = wide.merge(round_used, on="source_election_id", how="left")
    out = out.rename(columns={
        "candidate_1": "candidate_1_name_raw",
        "candidate_2": "candidate_2_name_raw",
        "party_1": "party_1_name_raw",
        "party_2": "party_2_name_raw",
        "share_1": "share_1",
        "share_2": "share_2",
    })
    out["share_metric"] = "vote_share"
    out["margin"] = out["share_1"] - out["share_2"]
    return out

ned_pres_path = paths["raw"] / "ned" / "presidential_elections_v2.dta"
ned_pres = pd.read_stata(ned_pres_path, convert_categoricals=False).reset_index()
ned_pres = ned_pres.rename(columns={"index": "source_election_id"})

pres_top2 = extract_top2_presidential(ned_pres)

# Construir tabla de eventos
pres_events = ned_pres[[
    "source_election_id",
    "country",
    "country_cow",
    "date",
    "year",
    "month",
    "type_election",
    "flag_two_round",
    "flag_inconsequential",
    "flag_coup",
    "flag_plebiscite",
    "flag_unopposed",
    "flag_indirect",
]].merge(pres_top2, on="source_election_id", how="left")

pres_events["election_date"] = pd.to_datetime(pres_events["date"], errors="coerce")
pres_events["election_year"] = pres_events["election_date"].dt.year.fillna(pres_events["year"]).astype("Int64")
pres_events["election_month"] = pres_events["election_date"].dt.month.fillna(pres_events["month"]).astype("Int64")
pres_events["election_day"] = pres_events["election_date"].dt.day.astype("Int64")

pres_events["date_precision"] = np.where(pres_events["election_date"].notna(), "day",
                                         np.where(pres_events["election_month"].notna(), "month", "year"))

pres_events["office_type"] = "presidential"
pres_events["source"] = "ned_pres"

pres_events.head()

,source_election_id,country,country_cow,date,year,month,type_election,flag_two_round,flag_inconsequential,flag_coup,flag_plebiscite,flag_unopposed,flag_indirect,candidate_1_name_raw,candidate_2_name_raw,party_1_name_raw,party_2_name_raw,share_1,share_2,round2_used,share_metric,margin,election_date,election_year,election_month,election_day,date_precision,office_type,source
0,0,Afghanistan,700.0,2004-10-09,2004,NaN,Presidential,NaN,NaN,NaN,NaN,NaN,NaN,Hamid Karzai,Yunus Qanuni,Independent,New Afghanistan Party,55.37,16.28,False,vote_share,39.09,2004-10-09,2004,10,9,day,presidential,ned_pres
1,1,Afghanistan,700.0,2009-08-20,2009,NaN,Presidential,NaN,NaN,NaN,NaN,NaN,NaN,Hamid Karzai,Abdullah Abdullah,Independent,National Coalition,49.67,30.59,False,vote_share,19.08,2009-08-20,2009,8,20,day,presidential,ned_pres
2,2,Afghanistan,700.0,2014-06-14,2014,NaN,Presidential,1.0,NaN,NaN,NaN,NaN,NaN,Ashraf Ghani Ahmadzai,Abdullah Abdullah,Independent,National Coalition of Afghanistan / Etelaf-e M...,56.44,43.56,True,vote_share,12.88,2014-06-14,2014,6,14,day,presidential,ned_pres
3,3,Afghanistan,700.0,2019-09-28,2019,NaN,Presidential,NaN,NaN,NaN,NaN,NaN,NaN,Mohammad Ashraf Ghani,Dr Abdullah Abdullah,Independent,National Coalition,50.6,39.5,False,vote_share,11.1,2019-09-28,2019,9,28,day,presidential,ned_pres
4,4,Algeria,615.0,1963-09-15,1963,NaN,Presidential,NaN,NaN,NaN,1.0,NaN,NaN,Ahmed Ben Bella,No votes,Front de Libération Nationale / National Liber...,,99.6,0.4,False,vote_share,99.2,1963-09-15,1963,9,15,day,presidential,ned_pres


## 6. NED parlamentario: top-2 y margenes (seat share)

Para elecciones parlamentarias, usamos `seat_share` como medida primaria (NED no incluye voto nacional comparable en todas las filas).

In [72]:
def extract_top2_parliamentary(df: pd.DataFrame) -> pd.DataFrame:
    party_nums = sorted({int(re.findall(r"party_(\d+)", c)[0]) for c in df.columns if c.startswith("party_")})
    parts = []
    for i in party_nums:
        party_col = f"party_{i}"
        share_col = f"seat_share_{i}"
        if share_col not in df.columns:
            continue
        tmp = df[["source_election_id", party_col, share_col]].copy()
        tmp.columns = ["source_election_id", "party", "share"]
        parts.append(tmp)

    long = pd.concat(parts, ignore_index=True)
    long = long[long["share"].notna()].copy()
    long = long.sort_values(["source_election_id", "share"], ascending=[True, False])
    long["rank"] = long.groupby("source_election_id").cumcount() + 1
    top2 = long[long["rank"] <= 2].copy()

    wide = top2.pivot(index="source_election_id", columns="rank", values=["party", "share"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    out = wide.rename(columns={
        "party_1": "party_1_name_raw",
        "party_2": "party_2_name_raw",
        "share_1": "share_1",
        "share_2": "share_2",
    })
    out["share_metric"] = "seat_share"
    out["margin"] = out["share_1"] - out["share_2"]
    return out

ned_parl_path = paths["raw"] / "ned" / "parliamentary_elections_v2.dta"
ned_parl = pd.read_stata(ned_parl_path, convert_categoricals=False).reset_index()
ned_parl = ned_parl.rename(columns={"index": "source_election_id"})

parl_top2 = extract_top2_parliamentary(ned_parl)

parl_events = ned_parl[[
    "source_election_id",
    "country",
    "country_cow",
    "date",
    "year",
    "month",
    "type_election",
    "flag_constituent",
    "flag_inconsequential",
    "flag_coup",
    "flag_vacant_seats",
    "flag_appointed",
    "flag_non_partisan",
]].merge(parl_top2, on="source_election_id", how="left")

parl_events["election_date"] = pd.to_datetime(parl_events["date"], errors="coerce")
parl_events["election_year"] = parl_events["election_date"].dt.year.fillna(parl_events["year"]).astype("Int64")
parl_events["election_month"] = parl_events["election_date"].dt.month.fillna(parl_events["month"]).astype("Int64")
parl_events["election_day"] = parl_events["election_date"].dt.day.astype("Int64")

parl_events["date_precision"] = np.where(parl_events["election_date"].notna(), "day",
                                         np.where(parl_events["election_month"].notna(), "month", "year"))

parl_events["office_type"] = "parliamentary"
parl_events["source"] = "ned_parl"

parl_events.head()

,source_election_id,country,country_cow,date,year,month,type_election,flag_constituent,flag_inconsequential,flag_coup,flag_vacant_seats,flag_appointed,flag_non_partisan,party_1_name_raw,party_2_name_raw,share_1,share_2,share_metric,margin,election_date,election_year,election_month,election_day,date_precision,office_type,source
0,0,Afghanistan,700.0,NaT,1931,NaN,Parliamentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,1931,<NA>,<NA>,year,parliamentary,ned_parl
1,1,Afghanistan,700.0,NaT,1934,NaN,Parliamentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,1934,<NA>,<NA>,year,parliamentary,ned_parl
2,2,Afghanistan,700.0,NaT,1937,NaN,Parliamentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,1937,<NA>,<NA>,year,parliamentary,ned_parl
3,3,Afghanistan,700.0,NaT,1940,NaN,Parliamentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,1940,<NA>,<NA>,year,parliamentary,ned_parl
4,4,Afghanistan,700.0,1943-04-01,1943,NaN,Parliamentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1943-04-01,1943,4,1,day,parliamentary,ned_parl


## 7. Unir NED y mapear ISO3

NED es la fuente primaria. Mapeamos ISO3 via COW con vigencia temporal y guardamos el metodo de match.

In [73]:
ned_events = pd.concat([pres_events, parl_events], ignore_index=True)

# Normalizar nombres de columnas
ned_events = ned_events.rename(columns={
    "country": "country_name_raw",
    "type_election": "type_election_raw",
})

# Mapear ISO3 con COW
iso_rows = ned_events.apply(lambda r: map_cow_to_iso(r["country_cow"], r["election_year"], cow), axis=1)
iso_df = pd.DataFrame(iso_rows.tolist(), columns=["iso3", "iso3_match_status"])
ned_events = pd.concat([ned_events, iso_df], axis=1)

# Overrides ISO3 por nombre (casos puntuales)
ned_iso3_overrides = {
    "Vietnam": "VNM",
    "Taiwan": "TWN",
    "Kosovo": "XKX",
    "Liechtenstein":"LIE",
    "Monaco":"MCO",
    "Netherlands Antilles":"ANT",
    "Serbia and Montenegro":"SCG",
    "Somaliland": "SOM"
}

override_mask = ned_events["country_name_raw"].isin(ned_iso3_overrides)
if override_mask.any():
    ned_events.loc[override_mask, "iso3"] = (
        ned_events.loc[override_mask, "country_name_raw"].map(ned_iso3_overrides)
    )
    ned_events.loc[override_mask, "iso3_match_status"] = "name_override"


# Fallback por nombre cuando falla COW
missing_mask = ned_events["iso3"].isna()
if missing_mask.any():
    ned_name_map_df = build_country_name_map(
        raw_names=ned_events.loc[missing_mask, "country_name_raw"].dropna().unique().tolist(),
        efw_name_map=efw_name_map,
        pycountry_map=pyc_map,
        manual_overrides=manual_overrides,
        fuzzy_threshold=90,
    )
    name_to_iso = ned_name_map_df.set_index("country_raw")["iso3"].to_dict()
    name_to_method = ned_name_map_df.set_index("country_raw")["match_method"].to_dict()

    ned_events.loc[missing_mask, "iso3"] = ned_events.loc[missing_mask, "country_name_raw"].map(name_to_iso)
    ned_events.loc[missing_mask, "iso3_match_status"] = (
        "name_" + ned_events.loc[missing_mask, "country_name_raw"].map(name_to_method).fillna("unmatched")
    )

ned_events["source_priority"] = 1

pd.DataFrame({
    "ned_rows": [len(ned_events)],
    "iso3_missing": [int(ned_events["iso3"].isna().sum())],
})


,ned_rows,iso3_missing
0,6309,0


## 8. CLEA: filtrar elecciones nacionales y agregar a nivel eleccion

CLEA es a nivel distrito/constituencia. Filtramos elecciones nacionales (`sub` = codigo negativo o missing), agregamos por eleccion y calculamos top-2.

In [74]:
clea_path = paths["raw"] / "clea" / "clea_lc_20251015.sav"
clea_cols = ["id", "ctr", "ctr_n", "yr", "mn", "sub", "pty", "pty_n", "pv1", "pvs1", "seat", "mag", "vot1"]
clea_raw, _ = pyreadstat.read_sav(clea_path, usecols=clea_cols) # Mostrar sin el segundo argumento

# Limpiar codigos negativos (missing)
for col in ["pv1", "pvs1", "seat", "mag", "vot1", "yr", "mn"]:
    clea_raw[col] = pd.to_numeric(clea_raw[col], errors="coerce")
    clea_raw.loc[clea_raw[col] < 0, col] = np.nan

# Filtrar a elecciones nacionales
sub_num = pd.to_numeric(clea_raw["sub"], errors="coerce")
mask_national = clea_raw["sub"].isna() | (sub_num <= -990)
clea_nat = clea_raw[mask_national].copy()

clea_nat.head()

,id,ctr_n,ctr,yr,mn,sub,mag,pty_n,pty,vot1,pv1,pvs1,seat
0,1.0,Botswana,72.0,1969.0,10.0,-990,1.0,botswana independence party,4.0,NaN,1390.0,0.351454,0.0
1,1.0,Botswana,72.0,1969.0,10.0,-990,1.0,botswana peoples party,6.0,NaN,987.0,0.249558,0.0
2,1.0,Botswana,72.0,1969.0,10.0,-990,1.0,botswana democratic party,3.0,NaN,1578.0,0.398989,1.0
3,1.0,Botswana,72.0,1969.0,10.0,-990,1.0,botswana independence party,4.0,NaN,1923.0,0.561788,1.0
4,1.0,Botswana,72.0,1969.0,10.0,-990,1.0,botswana democratic party,3.0,NaN,1500.0,0.438212,0.0


## 9. Mapeo de nombres de pais (CLEA -> ISO3)

Estrategia (orden):
1) match exacto/fuzzy con nombres EFW,
2) match con `pycountry`,
3) overrides manuales para casos ambiguos.

Todos los matches se guardan con metodo y score para auditoria.

In [75]:
# Mapas ya creados en la seccion 3.1

clea_country_map = build_country_name_map(
    raw_names=clea_nat["ctr_n"].dropna().unique().tolist(),
    efw_name_map=efw_name_map,
    pycountry_map=pyc_map,
    manual_overrides=manual_overrides,
    fuzzy_threshold=90,
)

# Diagnostico de no-matches
unmatched = clea_country_map[clea_country_map["iso3"].isna()]

clea_country_map.head(), unmatched.head()


(     country_raw iso3     match_method  match_score   matched_name
 0    Afghanistan  AFG  pycountry_exact        100.0    afghanistan
 1  Aland Islands  ALA  pycountry_exact        100.0  aland islands
 2        Albania  ALB        efw_exact        100.0        albania
 3        Algeria  DZA        efw_exact        100.0        algeria
 4        Andorra  AND  pycountry_exact        100.0        andorra,
     country_raw  iso3 match_method  match_score matched_name
 134  Somaliland  None    unmatched          NaN         None)

## 10. CLEA: top-2 y dataset de eventos

Agregamos votos y escaños por partido y eleccion. Para alinearnos con NED parlamentarias, usamos `seat_share` para rankear y calcular el margen (no `vote_share`).


In [76]:
# Agregar a nivel partido por eleccion
party = clea_nat.groupby(["id", "ctr", "ctr_n", "yr", "mn", "pty", "pty_n"], dropna=False).agg(
    pv1_sum=("pv1", "sum"),
    seat_sum=("seat", "sum"),
).reset_index()

# Totales por eleccion (usar suma agregada de asientos ganados)
keys = ["id", "ctr", "ctr_n", "yr", "mn"]
totals_votes = clea_nat.groupby(keys, dropna=False).agg(total_votes=("vot1", "max")).reset_index()
totals_seats = party.groupby(keys, dropna=False).agg(total_seats=("seat_sum", "sum")).reset_index()
totals = totals_votes.merge(totals_seats, on=keys, how="left")

party = party.merge(totals, on=keys, how="left")
party["vote_share"] = np.where(party["total_votes"] > 0, 100 * party["pv1_sum"] / party["total_votes"], np.nan)
party["seat_share"] = np.where(party["total_seats"] > 0, 100 * party["seat_sum"] / party["total_seats"], np.nan)

# Usamos seat_share para top-2 (alineado con NED parlamentarias)
party["share_for_rank"] = party["seat_share"]
party["share_metric"] = "seat_share"

# Top-2
party = party.sort_values(keys + ["share_for_rank"], ascending=[True, True, True, True, True, False])
party["rank"] = party.groupby(keys).cumcount() + 1
top2 = party[party["rank"] <= 2].copy()

wide = top2.pivot(index=keys, columns="rank", values=["pty", "pty_n", "share_for_rank"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide.columns = [c.replace(".0", "") for c in wide.columns]
wide = wide.reset_index()

# Metadatos de share (unico por eleccion)
share_meta = party.groupby(keys)["share_metric"].first().reset_index()

clea_events = wide.merge(share_meta, on=keys, how="left")
clea_events = clea_events.rename(columns={
    "pty_1": "party_1_clea_id",
    "pty_2": "party_2_clea_id",
    "pty_n_1": "party_1_name_raw",
    "pty_n_2": "party_2_name_raw",
    "share_for_rank_1": "share_1",
    "share_for_rank_2": "share_2",
    "yr": "election_year",
    "mn": "election_month",
})

for col in ["party_1_clea_id", "party_2_clea_id"]:
    clea_events[col] = pd.to_numeric(clea_events[col], errors="coerce").astype("Int64")

clea_events["margin"] = clea_events["share_1"] - clea_events["share_2"]
clea_events["office_type"] = "parliamentary"
clea_events["source"] = "clea_lc"
clea_events["source_election_id"] = clea_events["id"]

# Fechas (mes o anio)
clea_dates = clea_events.apply(lambda r: build_date_from_year_month(r["election_year"], r["election_month"]), axis=1)
clea_dates = pd.DataFrame(clea_dates.tolist(), columns=["election_date", "date_precision"])
clea_events = pd.concat([clea_events, clea_dates], axis=1)
clea_events["election_day"] = clea_events["election_date"].dt.day.astype("Int64")

# Mapear ISO3
clea_events = clea_events.merge(
    clea_country_map[["country_raw", "iso3", "match_method"]].rename(columns={"country_raw": "ctr_n", "match_method": "iso3_match_status"}),
    on="ctr_n",
    how="left",
)

clea_events = clea_events.rename(columns={
    "ctr_n": "country_name_raw",
    "ctr": "country_code_clea",
})

clea_events["source_priority"] = 2

clea_events.head()


,id,country_code_clea,country_name_raw,election_year,election_month,party_1_clea_id,party_2_clea_id,party_1_name_raw,party_2_name_raw,share_1,share_2,share_metric,margin,office_type,source,source_election_id,election_date,date_precision,election_day,iso3,iso3_match_status,source_priority
0,1.0,72.0,Botswana,1969.0,10.0,3,5,botswana democratic party,botswana national front,77.419355,9.677419,seat_share,67.741935,parliamentary,clea_lc,1.0,1969-10-01,month,1,BWA,efw_exact,2
1,2.0,72.0,Botswana,1974.0,10.0,3,5,botswana democratic party,botswana national front,84.375,6.25,seat_share,78.125,parliamentary,clea_lc,2.0,1974-10-01,month,1,BWA,efw_exact,2
2,3.0,72.0,Botswana,1979.0,10.0,3,5,botswana democratic party,botswana national front,90.625,6.25,seat_share,84.375,parliamentary,clea_lc,3.0,1979-10-01,month,1,BWA,efw_exact,2
3,4.0,72.0,Botswana,1984.0,9.0,3,5,botswana democratic party,botswana national front,85.294118,11.764706,seat_share,73.529412,parliamentary,clea_lc,4.0,1984-09-01,month,1,BWA,efw_exact,2
4,5.0,72.0,Botswana,1999.0,10.0,3,5,botswana democratic party,botswana national front,82.5,15.0,seat_share,67.5,parliamentary,clea_lc,5.0,1999-10-01,month,1,BWA,efw_exact,2


## 11. Union NED + CLEA y dedupe (prioridad NED)

Regla de dedupe:
1) Eliminamos CLEA con match estricto a NED (misma iso3, office, fecha exacta).
2) Eliminamos CLEA con match laxo a NED (misma iso3, office, anio+mes) solo cuando NED tiene un unico evento en ese anio+mes.

Esto preserva cobertura cuando NED es ambiguo o hay multiples elecciones en un mismo anio.

In [77]:
def build_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    iso = out["iso3"].fillna("UNK")
    office = out["office_type"].fillna("UNK")
    date_str = out["election_date"].dt.strftime("%Y-%m-%d")
    out["key_strict"] = np.where(out["election_date"].notna(), iso + "|" + office + "|" + date_str, np.nan)

    month = out["election_month"].fillna(0).astype(int).astype(str).str.zfill(2)
    year = out["election_year"].fillna(0).astype(int).astype(str)
    out["key_loose"] = iso + "|" + office + "|" + year + "|" + month
    return out

ned_k = build_keys(ned_events)
clea_k = build_keys(clea_events)

# Propagar IDs de partido CLEA a NED cuando hay match estricto (iso3+office+fecha)
clea_ids = clea_k[clea_k["key_strict"].notna()][["key_strict", "party_1_clea_id", "party_2_clea_id"]].copy()
key_counts = clea_ids["key_strict"].value_counts()
unique_keys = set(key_counts[key_counts == 1].index)
clea_ids = clea_ids[clea_ids["key_strict"].isin(unique_keys)]
ned_k = ned_k.merge(clea_ids, on="key_strict", how="left")

ned_strict = set(ned_k["key_strict"].dropna().unique().tolist())
clea_k["drop_strict"] = clea_k["key_strict"].isin(ned_strict)

ned_loose_counts = ned_k["key_loose"].value_counts()
unique_loose = set(ned_loose_counts[ned_loose_counts == 1].index)

# Propagar IDs de partido CLEA a NED cuando hay match laxo (iso3+office+anio+mes) y es unico
clea_ids_loose = clea_k[["key_loose", "party_1_clea_id", "party_2_clea_id"]].copy()
key_counts_loose = clea_ids_loose["key_loose"].value_counts()
unique_keys_loose = set(key_counts_loose[key_counts_loose == 1].index)
clea_ids_loose = clea_ids_loose[clea_ids_loose["key_loose"].isin(unique_keys_loose)]
clea_ids_loose = clea_ids_loose[clea_ids_loose["key_loose"].isin(unique_loose)]
ned_k = ned_k.merge(clea_ids_loose, on="key_loose", how="left", suffixes=("", "_loose"))
for col in ["party_1_clea_id", "party_2_clea_id"]:
    ned_k[col] = ned_k[col].combine_first(ned_k[f"{col}_loose"])
ned_k = ned_k.drop(columns=["party_1_clea_id_loose", "party_2_clea_id_loose"], errors="ignore")
clea_k["drop_loose"] = clea_k["key_loose"].isin(unique_loose)

clea_keep = clea_k[~(clea_k["drop_strict"] | clea_k["drop_loose"])].copy()

# Union final
events = pd.concat([ned_k, clea_keep], ignore_index=True)

# Limpiar columnas de keys intermedias
events = events.drop(columns=["key_strict", "key_loose", "drop_strict", "drop_loose"], errors="ignore")

# Eliminar filas sin margin antes del merge con PartyFacts
# events = events[events["margin"].notna()].copy()

# Flags EFW
events["efw_country"] = events["iso3"].isin(efw_iso3)
events["efw_post2000"] = events["efw_country"] & (events["election_year"] >= 2000)

pd.DataFrame({
    "ned_events": [len(ned_events)],
    "clea_events_raw": [len(clea_events)],
    "clea_events_kept": [len(clea_keep)],
    "events_total": [len(events)],
})

,ned_events,clea_events_raw,clea_events_kept,events_total
0,6309,1587,194,6503


### 11.1 Cobertura comparativa (NED vs NED+CLEA)

Este cuadro muestra la ganancia de cobertura 2000+ al incorporar CLEA sobre la base NED.

In [78]:
ned_2000 = ned_events[(ned_events["iso3"].isin(efw_iso3)) & (ned_events["election_year"] >= 2000)]
combined_2000 = events[events["efw_post2000"]]

ned_countries = set(ned_2000["iso3"].dropna())
combined_countries = set(combined_2000["iso3"].dropna())

coverage_gain = pd.DataFrame({
    "ned_countries_2000": [len(ned_countries)],
    "ned_events_2000": [len(ned_2000)],
    "combined_countries_2000": [len(combined_countries)],
    "combined_events_2000": [len(combined_2000)],
    "gain_countries": [len(combined_countries - ned_countries)],
})

coverage_gain


,ned_countries_2000,ned_events_2000,combined_countries_2000,combined_events_2000,gain_countries
0,159,1373,162,1415,3


## 12. Cobertura EFW (165 paises) y foco 2000+

Diagnosticos clave para presentacion: cobertura por pais en EFW y lista de faltantes 2000+.

In [79]:
# Cobertura por pais (2000+)
covered_2000 = events[events["efw_post2000"]].groupby("iso3").size().reset_index(name="n_elections")
missing_efw = sorted(set(efw_iso3) - set(covered_2000["iso3"]))

summary = pd.DataFrame({
    "efw_countries": [len(efw_iso3)],
    "covered_2000": [len(covered_2000)],
    "missing_2000": [len(missing_efw)],
    "events_2000": [int(events[events["efw_post2000"]].shape[0])],
})

summary, missing_efw[:20]

(   efw_countries  covered_2000  missing_2000  events_2000
 0            165           162             3         1415,
 ['BRN', 'CHN', 'SAU'])

## 13. Ideologia de partidos: insumos (PartyFacts + V-Party)

Objetivo: asignar ideologia economica al partido ganador (y runner-up) sin mezclar escalas.
- V-Party `v2pariglef` es la medida primaria.
- Ademas preservamos variables V-Party de populismo y liberal-tradicional (por ejemplo `v2xpa_*`, `ep_*`) y desempeno electoral (`v2paseatshare`, `v2pavote`).
- PartyFacts se usa solo como puente de IDs/nombres para maximizar el match de partidos.
- Priorizamos EFW y 2000+, pero construimos el enlace para todo el universo de eventos.


In [80]:
external_root = paths["root"] / "data" / "external"
pf_core_path = external_root / "partyfacts_core_parties.csv"
pf_external_path = external_root / "partyfacts_external_parties.csv"
vparty_path = paths["raw"] / "vparty" / "CPD_V-Party_CSV_v2" / "V-Dem-CPD-Party-V2.csv"

for p in [pf_core_path, pf_external_path, vparty_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing input file: {p}")

partyfacts_core = pd.read_csv(pf_core_path, low_memory=False)
partyfacts_external = pd.read_csv(pf_external_path, low_memory=False)
vparty_raw = pd.read_csv(vparty_path, low_memory=False)

# Normalizacion basica de paises/IDs
for df in [partyfacts_core, partyfacts_external]:
    df["country"] = df["country"].astype(str).str.upper().str.strip()
    df["partyfacts_id"] = pd.to_numeric(df["partyfacts_id"], errors="coerce").astype("Int64")

pd.DataFrame({
    "partyfacts_core_rows": [len(partyfacts_core)],
    "partyfacts_external_rows": [len(partyfacts_external)],
    "vparty_rows": [len(vparty_raw)],
})


,partyfacts_core_rows,partyfacts_external_rows,vparty_rows
0,7246,45385,11898


## 14. Construir alias de partidos (PartyFacts)

Creamos un diccionario de alias por pais usando PartyFacts core + external para robustecer el match de nombres.
Normalizamos nombres (sin tildes, minusculas, sin puntuacion) y conservamos la ventana temporal del partido.


In [81]:
GENERIC_PARTY_NAMES = {
    "independent", "independiente", "independents",
    "non partisan", "nonpartisan", "no party",
    "other", "others", "unknown", "n a", "na",
    "none", "null", "independent candidate",
}

def normalize_party_name(name: str) -> str:
    if pd.isna(name):
        return ""
    s = str(name)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    s = s.replace("&", " and ")
    s = re.sub(r"\(.*?\)|\[.*?\]", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = " ".join(s.split())
    return s

def split_aliases(name: str) -> list[str]:
    if pd.isna(name):
        return []
    raw = str(name).strip()
    if raw == "":
        return []
    parts = re.split(r"[|;]", raw)
    out = []
    for p in parts:
        p = p.strip()
        if p:
            out.append(p)
    return out

def build_party_aliases(pf_core: pd.DataFrame, pf_external: pd.DataFrame) -> pd.DataFrame:
    rows = []

    def push(row, alias, source):
        if alias is None or str(alias).strip() == "":
            return
        rows.append({
            "iso3": row.get("country"),
            "partyfacts_id": row.get("partyfacts_id"),
            "alias": alias,
            "alias_source": source,
            "technical": row.get("technical"),
            "year_first": row.get("year_first"),
            "year_last": row.get("year_last"),
            "share_year": row.get("share_year"),
        })

    core_cols = ["name_short", "name", "name_english", "name_other"]
    for _, row in pf_core.iterrows():
        for col in core_cols:
            for alias in split_aliases(row.get(col)):
                push(row, alias, f"core:{col}")

    ext_cols = ["name_short", "name", "name_english"]
    for _, row in pf_external.iterrows():
        dataset_key = row.get("dataset_key")
        for col in ext_cols:
            for alias in split_aliases(row.get(col)):
                push(row, alias, f"external:{dataset_key}:{col}")

    return pd.DataFrame(rows)

party_aliases = build_party_aliases(partyfacts_core, partyfacts_external)
party_aliases["partyfacts_id"] = pd.to_numeric(party_aliases["partyfacts_id"], errors="coerce").astype("Int64")
party_aliases["year_first"] = pd.to_numeric(party_aliases["year_first"], errors="coerce")
party_aliases["year_last"] = pd.to_numeric(party_aliases["year_last"], errors="coerce")
party_aliases["share_year"] = pd.to_numeric(party_aliases["share_year"], errors="coerce")
party_aliases["iso3"] = party_aliases["iso3"].astype(str).str.upper().str.strip()
party_aliases.loc[party_aliases["iso3"].isin(["", "NAN", "NONE"]), "iso3"] = np.nan
party_aliases = party_aliases.dropna(subset=["iso3", "partyfacts_id"])
party_aliases["alias_std"] = party_aliases["alias"].map(normalize_party_name)
party_aliases = party_aliases[party_aliases["alias_std"] != ""]
party_aliases = party_aliases[~party_aliases["alias_std"].isin(GENERIC_PARTY_NAMES)]
party_aliases = party_aliases.drop_duplicates(subset=["iso3", "partyfacts_id", "alias_std", "alias_source"])

ambiguous_aliases = (
    party_aliases.groupby(["iso3", "alias_std"])
    ["partyfacts_id"].nunique().reset_index(name="n_parties")
)
ambiguous_aliases = ambiguous_aliases[ambiguous_aliases["n_parties"] > 1]

pd.DataFrame({
    "aliases_total": [len(party_aliases)],
    "aliases_ambiguous": [len(ambiguous_aliases)],
})


,aliases_total,aliases_ambiguous
0,119403,1951


## 15. Match partidos de elecciones -> PartyFacts (prioridad ganador)

Se hace match por pais y anio, priorizando informacion deterministica:
0) si hay `party_1_clea_id`/`party_2_clea_id` (CLEA directo o propagado a NED por match estricto), se hace join directo a PartyFacts (dataset_key=clea).
1) coincidencia exacta de nombre normalizado,
2) fuzzy match si no hay exacto (umbral alto),
3) desempate por fuente (core > external), cercania temporal y `share_year` mas cercano al anio de eleccion.


In [82]:
# Mapeo deterministico CLEA -> PartyFacts (cuando existe codigo de partido)
pf_clea = partyfacts_external[partyfacts_external["dataset_key"] == "clea"].copy()
pf_clea["partyfacts_id"] = pd.to_numeric(pf_clea["partyfacts_id"], errors="coerce").astype("Int64")
pf_clea["pf_clea_id"] = pd.to_numeric(pf_clea["dataset_party_id"], errors="coerce").astype("Int64")
pf_clea["iso3"] = pf_clea["country"].astype(str).str.upper().str.strip()

pf_clea = pf_clea.dropna(subset=["pf_clea_id", "partyfacts_id", "iso3"])

# Prefijo por pais desde PartyFacts (modo del prefijo)
pf_clea["pf_prefix"] = (pf_clea["pf_clea_id"] // 1_000_000).astype("Int64")
prefix_map = (
    pf_clea.groupby("iso3")["pf_prefix"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NA)
    .to_dict()
)

# Fallback: ISO numerico estandar
iso_num_map = {}
for c in pycountry.countries:
    if hasattr(c, "alpha_3") and hasattr(c, "numeric"):
        iso_num_map[c.alpha_3] = int(c.numeric)

clea_pf_map = {
    int(row["pf_clea_id"]): int(row["partyfacts_id"])
    for _, row in pf_clea.drop_duplicates(subset=["pf_clea_id"]).iterrows()
}

def _encode_clea_id(iso3, clea_party_id):
    if pd.isna(iso3) or pd.isna(clea_party_id):
        return None
    try:
        clea_id = int(clea_party_id)
        if clea_id <= 0:
            return None
        iso = str(iso3).upper().strip()
        prefix = prefix_map.get(iso)
        if pd.isna(prefix) or prefix is None:
            prefix = iso_num_map.get(iso)
        if prefix is None:
            return None
        return int(prefix) * 1_000_000 + clea_id
    except Exception:
        return None

# Codificar IDs CLEA en events para evitar confusion (guardar raw)
for col in ["party_1_clea_id", "party_2_clea_id"]:
    raw_col = f"{col}_raw"
    if raw_col not in events.columns:
        events[raw_col] = events[col]
    events[col] = events.apply(lambda r: _encode_clea_id(r["iso3"], r[col]), axis=1)
    events[col] = pd.to_numeric(events[col], errors="coerce").astype("Int64")

def map_clea_partyfacts(iso3, clea_party_id):
    if pd.isna(clea_party_id):
        return None
    try:
        cid = int(clea_party_id)
    except Exception:
        return None
    if cid <= 0:
        return None
    # Si ya esta codificado (o existe en el mapa), usarlo directo
    if cid in clea_pf_map:
        return clea_pf_map.get(cid)
    pf_id = _encode_clea_id(iso3, cid)
    if pf_id is None:
        return None
    return clea_pf_map.get(pf_id)

override_path = paths["manual"] / "party_match_overrides.csv"
overrides = {}
if override_path.exists():
    manual = pd.read_csv(override_path)
    if not manual.empty:
        manual["iso3"] = manual["iso3"].astype(str).str.upper().str.strip()
        manual["election_year"] = pd.to_numeric(manual["election_year"], errors="coerce")
        manual["party_name_std"] = manual["party_name_raw"].map(normalize_party_name)
        for _, r in manual.dropna(subset=["iso3", "party_name_std", "partyfacts_id"]).iterrows():
            key = (r["iso3"], int(r["election_year"]) if pd.notna(r["election_year"]) else None, r["party_name_std"])
            overrides[key] = int(r["partyfacts_id"])

MATCH_YEAR_WINDOW = 5
FUZZY_THRESHOLD = 90
TIE_DELTA = 3

def _filter_pool(aliases: pd.DataFrame, iso3: str | None, election_year: int | None, year_window: int | None) -> pd.DataFrame:
    if iso3 is None or pd.isna(iso3):
        return aliases
    pool = aliases[aliases["iso3"] == iso3]
    if election_year is None or pd.isna(election_year):
        return pool
    y = int(election_year)
    if year_window is None:
        return pool[(pool["year_first"].isna() | (pool["year_first"] <= y)) & (pool["year_last"].isna() | (pool["year_last"] >= y))]
    return pool[(pool["year_first"].isna() | (pool["year_first"] <= y + year_window)) & (pool["year_last"].isna() | (pool["year_last"] >= y - year_window))]

def _is_name_other_source(source: str | None) -> bool:
    return str(source).startswith("core:name_other")
def _tiebreak_candidates(cand: pd.DataFrame, election_year: int | None) -> tuple[int | None, str]:
    cand = cand.dropna(subset=["partyfacts_id"]).copy()
    if cand.empty:
        return None, "no_candidates"
    cand["technical_rank"] = cand.get("technical").map({False: 0, True: 1}).fillna(2)
    cand["source_rank"] = cand.get("alias_source").map(lambda s: 0 if str(s).startswith("core") else 1).fillna(2)
    cand["alias_priority"] = cand.get("alias_source").map(lambda s: 1 if _is_name_other_source(s) else 0).fillna(0)

    if election_year is not None and not pd.isna(election_year):
        y = float(election_year)
        yf = pd.to_numeric(cand.get("year_first"), errors="coerce")
        yl = pd.to_numeric(cand.get("year_last"), errors="coerce")
        mid = (yf + yl) / 2.0
        cand["year_mid_gap"] = (mid - y).abs()
        cand["year_span"] = (yl - yf).abs().fillna(float("inf"))
        cand["in_range"] = (yf <= y) & (yl >= y)
        cand["share_year_gap"] = (pd.to_numeric(cand.get("share_year"), errors="coerce") - y).abs()
    else:
        cand["year_mid_gap"] = float("inf")
        cand["year_span"] = float("inf")
        cand["in_range"] = False
        cand["share_year_gap"] = float("inf")

    cand = cand.sort_values(
        ["in_range", "technical_rank", "source_rank", "alias_priority", "share_year_gap", "year_mid_gap", "year_span", "partyfacts_id"],
        ascending=[False, True, True, True, True, True, True, True],
        kind="mergesort",
    )
    if len(cand) > 1:
        top = cand.iloc[0]
        second = cand.iloc[1]
        tie_cols = ["in_range", "technical_rank", "source_rank", "alias_priority", "share_year_gap", "year_mid_gap", "year_span"]
        if all(top[c] == second[c] for c in tie_cols):
            return None, "ambiguous"
    return int(cand.iloc[0]["partyfacts_id"]), "tiebreak"
def match_party(name_raw, iso3, election_year, aliases: pd.DataFrame, overrides: dict, direct_partyfacts_id: int | None = None) -> dict:
    name_std = normalize_party_name(name_raw)

    iso3_norm = None if pd.isna(iso3) else str(iso3).upper().strip()
    year_norm = None
    if election_year is not None and not pd.isna(election_year):
        try:
            year_norm = int(election_year)
        except Exception:
            year_norm = None

    key = (iso3_norm, year_norm, name_std)
    if overrides and key in overrides:
        return {
            "party_name_std": name_std,
            "partyfacts_id": overrides[key],
            "match_status": "matched_manual",
            "match_method": "manual",
            "match_score": 100,
            "match_alias": name_std,
            "match_alias_source": "manual",
            "match_tie": False,
            "candidate_ids": str(overrides[key]),
        }

    if direct_partyfacts_id is not None and not pd.isna(direct_partyfacts_id):
        return {
            "party_name_std": name_std,
            "partyfacts_id": int(direct_partyfacts_id),
            "match_status": "matched_clea_id",
            "match_method": "clea_id",
            "match_score": 100,
            "match_alias": None,
            "match_alias_source": "clea_id",
            "match_tie": False,
            "candidate_ids": str(int(direct_partyfacts_id)),
        }

    if name_std == "":
        return {
            "party_name_std": name_std,
            "partyfacts_id": None,
            "match_status": "missing",
            "match_method": "none",
            "match_score": None,
            "match_alias": None,
            "match_alias_source": None,
            "match_tie": False,
            "candidate_ids": "",
        }
    if name_std in GENERIC_PARTY_NAMES:
        return {
            "party_name_std": name_std,
            "partyfacts_id": None,
            "match_status": "generic",
            "match_method": "none",
            "match_score": None,
            "match_alias": None,
            "match_alias_source": None,
            "match_tie": False,
            "candidate_ids": "",
        }

    pool = _filter_pool(aliases, iso3_norm, year_norm, MATCH_YEAR_WINDOW)
    if pool.empty and iso3_norm is not None:
        pool = aliases[aliases["iso3"] == iso3_norm]
    if pool.empty:
        return {
            "party_name_std": name_std,
            "partyfacts_id": None,
            "match_status": "no_candidates",
            "match_method": "none",
            "match_score": None,
            "match_alias": None,
            "match_alias_source": None,
            "match_tie": False,
            "candidate_ids": "",
        }

    pool_pref = pool
    has_name_other = False
    if "alias_source" in pool.columns:
        pool_pref = pool[~pool["alias_source"].map(_is_name_other_source)]
        has_name_other = len(pool_pref) < len(pool)
        if pool_pref.empty:
            pool_pref = pool
            has_name_other = False

    def _attempt_exact(pool_in: pd.DataFrame) -> dict | None:
        exact = pool_in[pool_in["alias_std"] == name_std]
        if exact.empty:
            return None
        pid, status = _tiebreak_candidates(exact, year_norm)
        if pid is None:
            cands = sorted({int(x) for x in exact["partyfacts_id"].dropna().unique().tolist()})
            return {
                "party_name_std": name_std,
                "partyfacts_id": None,
                "match_status": "ambiguous_exact",
                "match_method": "exact",
                "match_score": 100,
                "match_alias": name_std,
                "match_alias_source": None,
                "match_tie": True,
                "candidate_ids": "|".join(str(x) for x in cands),
            }
        alias_source = exact.loc[exact["partyfacts_id"] == pid, "alias_source"].iloc[0] if not exact.empty else None
        return {
            "party_name_std": name_std,
            "partyfacts_id": pid,
            "match_status": "matched_exact",
            "match_method": "exact",
            "match_score": 100,
            "match_alias": name_std,
            "match_alias_source": alias_source,
            "match_tie": False,
            "candidate_ids": str(pid),
        }

    def _attempt_fuzzy(pool_in: pd.DataFrame, allow_unmatched: bool) -> dict | None:
        choices = pool_in["alias_std"].dropna().unique().tolist()
        results = process.extract(name_std, choices, scorer=fuzz.token_sort_ratio, limit=2)
        if not results:
            if not allow_unmatched:
                return None
            return {
                "party_name_std": name_std,
                "partyfacts_id": None,
                "match_status": "unmatched",
                "match_method": "fuzzy",
                "match_score": None,
                "match_alias": None,
                "match_alias_source": None,
                "match_tie": False,
                "candidate_ids": "",
            }

        best_alias, score1, _ = results[0]
        second = results[1] if len(results) > 1 else None
        score2 = second[1] if second else None
        if score1 < FUZZY_THRESHOLD:
            if not allow_unmatched:
                return None
            return {
                "party_name_std": name_std,
                "partyfacts_id": None,
                "match_status": "unmatched",
                "match_method": "fuzzy",
                "match_score": int(score1),
                "match_alias": best_alias,
                "match_alias_source": None,
                "match_tie": False,
                "candidate_ids": "",
            }

        tie = False
        if score2 is not None and (score1 - score2) <= TIE_DELTA:
            tie = True

        cand = pool_in[pool_in["alias_std"] == best_alias]
        pid, status = _tiebreak_candidates(cand, year_norm)
        if pid is None:
            cands = sorted({int(x) for x in cand["partyfacts_id"].dropna().unique().tolist()})
            return {
                "party_name_std": name_std,
                "partyfacts_id": None,
                "match_status": "ambiguous_fuzzy",
                "match_method": "fuzzy",
                "match_score": int(score1),
                "match_alias": best_alias,
                "match_alias_source": None,
                "match_tie": tie,
                "candidate_ids": "|".join(str(x) for x in cands),
            }

        alias_source = cand.loc[cand["partyfacts_id"] == pid, "alias_source"].iloc[0] if not cand.empty else None
        return {
            "party_name_std": name_std,
            "partyfacts_id": pid,
            "match_status": "matched_fuzzy_tie" if tie else "matched_fuzzy",
            "match_method": "fuzzy",
            "match_score": int(score1),
            "match_alias": best_alias,
            "match_alias_source": alias_source,
            "match_tie": tie,
            "candidate_ids": str(pid),
        }

    res = _attempt_exact(pool_pref)
    if res is not None:
        return res
    if has_name_other:
        res = _attempt_exact(pool)
        if res is not None:
            return res

    res = _attempt_fuzzy(pool_pref, allow_unmatched=not has_name_other)
    if res is not None:
        return res
    if has_name_other:
        return _attempt_fuzzy(pool, allow_unmatched=True)
    return _attempt_fuzzy(pool, allow_unmatched=True)

events = events.reset_index(drop=True).copy()
events["event_id"] = np.arange(len(events))

party_long = events[[
    "event_id", "iso3", "election_year", "source", "office_type",
    "party_1_name_raw", "party_2_name_raw",
]].melt(
    id_vars=["event_id", "iso3", "election_year", "source", "office_type"],
    value_vars=["party_1_name_raw", "party_2_name_raw"],
    var_name="party_role_raw",
    value_name="party_name_raw",
)
party_long["party_role"] = party_long["party_role_raw"].map({"party_1_name_raw": "winner", "party_2_name_raw": "runnerup"})
party_long = party_long.drop(columns=["party_role_raw"])

party_long = party_long.merge(
    events[["event_id", "party_1_clea_id", "party_2_clea_id"]],
    on="event_id",
    how="left",
)
party_long["clea_party_id"] = np.where(
    party_long["party_role"] == "winner",
    party_long["party_1_clea_id"],
    party_long["party_2_clea_id"],
)
party_long = party_long.drop(columns=["party_1_clea_id", "party_2_clea_id"])

unique_parties = party_long[["iso3", "election_year", "party_name_raw", "clea_party_id"]].drop_duplicates()
unique_parties["direct_partyfacts_id"] = unique_parties.apply(
    lambda r: map_clea_partyfacts(r["iso3"], r["clea_party_id"]), axis=1
)
match_df = unique_parties.apply(
    lambda r: pd.Series(match_party(
        r["party_name_raw"], r["iso3"], r["election_year"], party_aliases, overrides,
        direct_partyfacts_id=r["direct_partyfacts_id"],
    )),
    axis=1,
)
unique_parties = pd.concat([unique_parties, match_df], axis=1)

party_long = party_long.merge(
    unique_parties,
    on=["iso3", "election_year", "party_name_raw", "clea_party_id"],
    how="left",
)

party_match_audit = party_long[[
    "event_id", "iso3", "election_year", "source", "office_type", "party_role", "clea_party_id",
    "party_name_raw", "party_name_std",
    "partyfacts_id", "match_status", "match_method", "match_score",
    "match_alias", "match_alias_source", "match_tie", "candidate_ids",
]].copy()

cols_to_wide = ["partyfacts_id", "party_name_std", "match_status", "match_method", "match_score", "match_alias", "match_alias_source", "match_tie"]
party_wide = party_match_audit.set_index(["event_id", "party_role"])[cols_to_wide].unstack("party_role")
party_wide.columns = [f"{c}_{role}" for c, role in party_wide.columns]
party_wide = party_wide.reset_index()

events = events.merge(party_wide, on="event_id", how="left")
events = events.rename(columns={
    "partyfacts_id_winner": "winner_partyfacts_id",
    "partyfacts_id_runnerup": "runnerup_partyfacts_id",
    "party_name_std_winner": "winner_party_name_std",
    "party_name_std_runnerup": "runnerup_party_name_std",
    "match_status_winner": "winner_party_match_status",
    "match_status_runnerup": "runnerup_party_match_status",
    "match_method_winner": "winner_party_match_method",
    "match_method_runnerup": "runnerup_party_match_method",
    "match_score_winner": "winner_party_match_score",
    "match_score_runnerup": "runnerup_party_match_score",
    "match_alias_winner": "winner_party_match_alias",
    "match_alias_runnerup": "runnerup_party_match_alias",
    "match_alias_source_winner": "winner_party_match_alias_source",
    "match_alias_source_runnerup": "runnerup_party_match_alias_source",
    "match_tie_winner": "winner_party_match_tie",
    "match_tie_runnerup": "runnerup_party_match_tie",
})

for col in ["winner_partyfacts_id", "runnerup_partyfacts_id"]:
    events[col] = pd.to_numeric(events[col], errors="coerce").astype("Int64")

pd.DataFrame({
    "events_total": [len(events)],
    "winner_partyfacts_matched": [events["winner_partyfacts_id"].notna().sum()],
    "runnerup_partyfacts_matched": [events["runnerup_partyfacts_id"].notna().sum()],
})


,events_total,winner_partyfacts_matched,runnerup_partyfacts_matched
0,6503,3433,2884


## 16. Ideologia V-Party por partido y anio (sin mezclar escalas)

Tomamos `v2pariglef` como escala principal y conservamos otras variables V-Party (p. ej., `v2xpa_*`, `ep_*`, `v2paseatshare`, `v2pavote`). Para cada partido, usamos el valor del anio de la eleccion cuando existe; si no, el anio mas cercano y registramos la brecha temporal.
Esto permite maximizar cobertura manteniendo trazabilidad (se puede filtrar por una ventana recomendada).


In [83]:
VPA_METRICS = [
    "v2pariglef",
    "v2xpa_antiplural",
    "v2xpa_popul",
    "ep_antielite_salience",
    "ep_corrupt_salience",
    "ep_members_vs_leadership",
    "ep_people_vs_elite",
    "ep_type_populism",
    "ep_type_populist_values",
    "ep_v8_popul_rhetoric",
    "ep_v9_popul_saliency",
    "ep_galtan",
    "ep_galtan_salience",
    "ep_v6_lib_cons",
    "ep_v7_lib_cons_saliency",
    "v2paseatshare",
    "v2pavote",
]

vparty_obs = vparty_raw[["pf_party_id", "year"] + VPA_METRICS].copy()
vparty_obs["partyfacts_id"] = pd.to_numeric(vparty_obs["pf_party_id"], errors="coerce").astype("Int64")
vparty_obs["year"] = pd.to_numeric(vparty_obs["year"], errors="coerce").astype("Int64")
for c in VPA_METRICS:
    vparty_obs[c] = pd.to_numeric(vparty_obs[c], errors="coerce")

vparty_obs = vparty_obs.dropna(subset=["partyfacts_id", "year"])
vparty_obs = vparty_obs[vparty_obs[VPA_METRICS].notna().any(axis=1)]

# Si hubiera duplicados party-year, promediamos en la misma escala (no mezcla de medidas)
vparty_obs = vparty_obs.groupby(["partyfacts_id", "year"], as_index=False)[VPA_METRICS].mean()

IDEO_MAX_GAP = 5  # ventana recomendada (anios) para considerar ideologia contemporanea

party_ideo = party_long.merge(vparty_obs, on="partyfacts_id", how="left")
party_ideo["year_gap"] = (party_ideo["election_year"] - party_ideo["year"]).abs()
party_ideo["year_before"] = party_ideo["year"] <= party_ideo["election_year"]

party_ideo = party_ideo.sort_values(
    ["event_id", "party_role", "year_gap", "year_before", "year"],
    ascending=[True, True, True, False, False],
)

party_ideo_best = party_ideo.groupby(["event_id", "party_role"], as_index=False).first()
party_ideo_best["ideo_ok"] = (party_ideo_best["year_gap"] <= IDEO_MAX_GAP) & party_ideo_best["v2pariglef"].notna()
party_ideo_best["ideo_source"] = np.where(party_ideo_best["v2pariglef"].notna(), "vparty_v2pariglef", pd.NA)

ideo_cols = VPA_METRICS + ["year", "year_gap", "ideo_ok", "ideo_source"]
ideo_wide = party_ideo_best.set_index(["event_id", "party_role"])[ideo_cols].unstack("party_role")
ideo_wide.columns = [f"{c}_{role}" for c, role in ideo_wide.columns]
ideo_wide = ideo_wide.reset_index()

events = events.merge(ideo_wide, on="event_id", how="left")
rename_map = {
    "v2pariglef_winner": "winner_ideo_vparty",
    "v2pariglef_runnerup": "runnerup_ideo_vparty",
    "year_winner": "winner_ideo_year",
    "year_runnerup": "runnerup_ideo_year",
    "year_gap_winner": "winner_ideo_gap",
    "year_gap_runnerup": "runnerup_ideo_gap",
    "ideo_ok_winner": "winner_ideo_ok",
    "ideo_ok_runnerup": "runnerup_ideo_ok",
    "ideo_source_winner": "winner_ideo_source",
    "ideo_source_runnerup": "runnerup_ideo_source",
}
for c in VPA_METRICS:
    if c == "v2pariglef":
        continue
    rename_map[f"{c}_winner"] = f"winner_vparty_{c}"
    rename_map[f"{c}_runnerup"] = f"runnerup_vparty_{c}"

events = events.rename(columns=rename_map)

for col in ["winner_ideo_year", "runnerup_ideo_year"]:
    events[col] = pd.to_numeric(events[col], errors="coerce").astype("Int64")

pd.DataFrame({
    "winner_ideo_available": [events["winner_ideo_vparty"].notna().sum()],
    "runnerup_ideo_available": [events["runnerup_ideo_vparty"].notna().sum()],
})


,winner_ideo_available,runnerup_ideo_available
0,2333,1916


## 17. Cobertura ideologica (EFW 2000+, foco ganador)

Diagnosticos de cobertura para el universo EFW desde 2000.
Separamos: match a PartyFacts, ideologia V-Party disponible y dentro de la ventana recomendada.


In [20]:
efw2000 = events[events["efw_post2000"]].copy()

coverage_winner = pd.DataFrame({
    "events_efw2000": [len(efw2000)],
    "winner_name_missing": [efw2000["party_1_name_raw"].isna().sum()],
    "winner_partyfacts_matched": [efw2000["winner_partyfacts_id"].notna().sum()],
    "winner_vparty_available": [efw2000["winner_ideo_vparty"].notna().sum()],
    "winner_vparty_in_window": [((efw2000["winner_ideo_vparty"].notna()) & (efw2000["winner_ideo_ok"].fillna(False))).sum()],
})

coverage_winner


,events_efw2000,winner_name_missing,winner_partyfacts_matched,winner_vparty_available,winner_vparty_in_window
0,1415,22,908,798,768


In [21]:
country_cov = (
    efw2000.groupby("iso3").agg(
        n_events=("event_id", "size"),
        winner_ideo=("winner_ideo_vparty", "count"),
        winner_ideo_ok=("winner_ideo_ok", lambda s: int(s.fillna(False).sum())),
    ).reset_index()
)
country_cov["winner_ideo_rate"] = (country_cov["winner_ideo"] / country_cov["n_events"]).round(3)

missing_ideo_countries = country_cov[country_cov["winner_ideo"] == 0]["iso3"].tolist()

country_cov.sort_values("winner_ideo_rate").head(15), missing_ideo_countries[:20]


(    iso3  n_events  winner_ideo  winner_ideo_ok  winner_ideo_rate
 27   CHL        13            0               0               0.0
 18   BLZ         5            0               0               0.0
 17   BLR        11            0               0               0.0
 82   KWT        12            0               0               0.0
 15   BHS         5            0               0               0.0
 47   ETH         5            0               0               0.0
 157  VNM         5            0               0               0.0
 139  SWZ         5            0               0               0.0
 133  SOM         5            0               0               0.0
 75   JOR         6            0               0               0.0
 106  MYS         5            0               0               0.0
 124  QAT         1            0               0               0.0
 115  OMN         7            0               0               0.0
 120  PNG         7            0               0              

## 18. Export final

Se guardan:
- `elections_master.parquet`: eventos de elecciones (todas las fuentes, con dedupe) + IDs PartyFacts e ideologia V-Party.
- `elections_master_efw2000.parquet`: subset con paises EFW y anio >= 2000 (incluye ideologia).
- `clea_country_map.csv`: mapeo de nombres CLEA -> ISO3 (auditable).
- `party_match_results.csv`: auditoria del match de partidos (ganador y runner-up).


In [22]:
out_all = paths["processed"] / "elections_master.parquet"
out_efw = paths["processed"] / "elections_master_efw2000.parquet"
out_map = paths["processed"] / "clea_country_map.csv"

events.to_parquet(out_all, index=False)

events[events["efw_post2000"]].to_parquet(out_efw, index=False)

clea_country_map.to_csv(out_map, index=False)

out_all, out_efw, out_map

(PosixPath('/Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/elections_master.parquet'),
 PosixPath('/Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/elections_master_efw2000.parquet'),
 PosixPath('/Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/clea_country_map.csv'))

## 19. DPI 2020: ideología económica y fragmentación/polarización (merge con `events`)

DPI es un panel país-año; `events` es un dataset de elecciones (fecha específica). Para observar la ideología **antes** y **después** de la elección:

- **País**: mapear `countryname` -> ISO3 con el mismo procedimiento usado en CLEA (EFW + pycountry + overrides), y revisar no-matches.
- **Ideología**: estandarizar R/L/C a códigos {Right=1, Center=2, Left=3, 0=sin info/no encaja, NA=sin ejecutivo}; tratar `-999` como missing.
- **Tiempo**: para cada variable DPI se crean dos variantes respecto al año de la elección:
  - `*_pre1`: DPI del **año anterior** (`election_year - 1`)
  - `*_post1`: DPI del **año siguiente** (`election_year + 1`)
  (sin regla H1/H2; el merge usa solo `iso3` + año).
- **Trazabilidad**: conservar `dpi_year_pre1` y `dpi_year_post1`.


In [23]:
dpi_path = paths["root"] / "DPI2020.csv"
dpi = pd.read_csv(dpi_path, low_memory=False)

# Columnas de interes
rlc_cols = ["execrlc", "gov1rlc", "gov2rlc", "gov3rlc", "opp1rlc"]
frag_cols = ["herfgov", "herfopp", "herftot", "govfrac", "oppfrac", "frac", "polariz"]

# Vista rapida del contenido relevante
preview_cols = ["countryname", "ifs", "year"] + rlc_cols + frag_cols
dpi[preview_cols].head()


,countryname,ifs,year,execrlc,gov1rlc,gov2rlc,gov3rlc,opp1rlc,herfgov,herfopp,herftot,govfrac,oppfrac,frac,polariz
0,Turk Cyprus,0,1975,0,-999,-999,-999,-999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Turk Cyprus,0,1976,0,-999,-999,-999,-999,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,Turk Cyprus,0,1977,Right,0,-999,-999,Left,1.0,0.371901,NaN,0.0,0.690909,0.457692,0.0
3,Turk Cyprus,0,1978,Right,0,-999,-999,Left,1.0,0.371901,NaN,0.0,0.690909,0.457692,0.0
4,Turk Cyprus,0,1979,Right,0,-999,-999,Left,1.0,0.371901,NaN,0.0,0.690909,0.457692,0.0


In [24]:
# 1) Mapear countryname -> ISO3
# Si aparecen no-matches en dpi_unmatched, agregar overrides en manual_overrides

dpi["countryname"] = dpi["countryname"].astype(str).str.strip()

# Construir mapa DPI -> ISO3 (reutiliza helpers del notebook)
dpi_country_map = build_country_name_map(
    raw_names=dpi["countryname"].dropna().unique().tolist(),
    efw_name_map=efw_name_map,
    pycountry_map=pyc_map,
    manual_overrides=manual_overrides,
    fuzzy_threshold=90,
)

dpi = dpi.merge(
    dpi_country_map.rename(columns={"country_raw": "countryname"}),
    on="countryname",
    how="left",
)

# 2) Recodificar ideologia R/L/C
RLC_STR_MAP = {"RIGHT": 1, "CENTER": 2, "CENTRE": 2, "LEFT": 3}
MISSING_CODES = {"-999", "-998", "-997", "-990"}

def recode_rlc(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip()
    if s == "":
        return pd.NA
    if s in MISSING_CODES:
        return pd.NA
    su = s.upper()
    if su in RLC_STR_MAP:
        return RLC_STR_MAP[su]
    if su in {"0", "NO INFO", "NONE"}:
        return 0
    try:
        n = int(float(s))
        if n in {-999, -998, -997, -990}:
            return pd.NA
        if n in {0, 1, 2, 3}:
            return n
    except Exception:
        return pd.NA
    return pd.NA

for col in rlc_cols:
    dpi[f"{col}_code"] = dpi[col].apply(recode_rlc).astype("Int64")

# Diagnostico de no-matches
# (si hay casos relevantes, agregar overrides y re-ejecutar)
dpi_unmatched = (
    dpi[dpi["iso3"].isna()][["countryname"]]
    .drop_duplicates()
    .sort_values("countryname")
)

# Diagnostico de duplicados por iso3-year
n_dupes = dpi.duplicated(subset=["iso3", "year"]).sum()

dpi_unmatched.head(15), n_dupes


(         countryname
 828      Bosnia-Herz
 1104          Brunei
 1702    C. Verde Is.
 1242  Cent. Af. Rep.
 1656      Comoro Is.
 8062     Congo (DRC)
 1794      Czech Rep.
 2116       Dom. Rep.
 2944      Eq. Guinea
 1978     FRG/Germany
 1932             GDR
 5888    P. N. Guinea
 1426             PRC
 5980             PRK
 4048             ROK,
 1165)

In [ ]:
# 3) Preparar subset DPI para merge con events (pre/post)

keep_raw = ["countryname", "ifs", "year"] + rlc_cols + frag_cols
keep_codes = [f"{c}_code" for c in rlc_cols]

# Nota: dpi ya tiene iso3 mapeado

# Normalizar iso3 y year antes de deduplicar/merge
dpi["iso3"] = dpi["iso3"].astype(str).str.upper().str.strip()
events["iso3"] = events["iso3"].astype(str).str.upper().str.strip()
dpi["year"] = pd.to_numeric(dpi["year"], errors="coerce").astype("Int64")

# Diagnostico de duplicados por iso3-year (antes de colapsar)
_dpi_dupes = dpi.dropna(subset=["iso3"]).duplicated(subset=["iso3", "year"]).sum()
# Diagnostico de no-matches (si hay casos relevantes, agregar overrides y re-ejecutar)
dpi_unmatched = (
    dpi[dpi["iso3"].isna()][["countryname"]]
    .drop_duplicates()
    .sort_values("countryname")
)

# Diagnostico de duplicados por iso3-year (antes de colapsar)

# Subset DPI con columnas relevantes

dpi_keep = dpi[["iso3"] + keep_raw + keep_codes].copy()

# Evitar matches espurios con iso3 nulo

dpi_keep = dpi_keep[dpi_keep["iso3"].notna()].copy()

# Prefijar columnas y estandarizar nombre de año

dpi_keep = dpi_keep.rename(columns={
    **{c: f"dpi_{c}" for c in dpi_keep.columns if c not in {"iso3", "year"}},
    "year": "dpi_year",
})

# Si hubiera duplicados iso3-year, colapsar conservando la fila más informativa

dpi_cols = [c for c in dpi_keep.columns if c not in {"iso3", "dpi_year"}]
if _dpi_dupes:
    dpi_keep["_non_nulls"] = dpi_keep[dpi_cols].notna().sum(axis=1)
    dpi_keep = (
        dpi_keep.sort_values(["iso3", "dpi_year", "_non_nulls"], ascending=[True, True, False])
        .drop_duplicates(subset=["iso3", "dpi_year"], keep="first")
        .drop(columns=["_non_nulls"])
    )

# 4) Construir variantes pre/post por año de elección ///// IMPORTANTE: esto lo hago para evitar problemas con el año de elección.

dpi_pre = dpi_keep.rename(columns={
    **{c: f"{c}_pre1" for c in dpi_cols},
    "dpi_year": "dpi_year_pre1",
})

dpi_post = dpi_keep.rename(columns={
    **{c: f"{c}_post1" for c in dpi_cols},
    "dpi_year": "dpi_year_post1",
})

# 5) Merge por iso3 + año (solo país y año)

events_dpi = events.copy()
_events_year = pd.to_numeric(events_dpi["election_year"], errors="coerce").astype("Int64")

events_dpi["dpi_year_pre1"] = _events_year - 1

events_dpi["dpi_year_post1"] = _events_year + 1

# Merge pre y post

events_dpi = events_dpi.merge(
    dpi_pre,
    on=["iso3", "dpi_year_pre1"],
    how="left",
)

events_dpi = events_dpi.merge(
    dpi_post,
    on=["iso3", "dpi_year_post1"],
    how="left",
)

# Diagnostico de cobertura

coverage_dpi = pd.DataFrame({
    "events_total": [len(events_dpi)],
    "events_with_dpi_pre1": [events_dpi["dpi_countryname_pre1"].notna().sum()],
    "events_with_dpi_post1": [events_dpi["dpi_countryname_post1"].notna().sum()],
    "execrlc_pre1_available": [events_dpi["dpi_execrlc_code_pre1"].notna().sum()],
    "execrlc_post1_available": [events_dpi["dpi_execrlc_code_post1"].notna().sum()],
    "polariz_pre1_available": [events_dpi["dpi_polariz_pre1"].notna().sum()],
    "polariz_post1_available": [events_dpi["dpi_polariz_post1"].notna().sum()],
})

coverage_dpi


,events_total,events_with_dpi_pre1,events_with_dpi_post1,execrlc_pre1_available,execrlc_post1_available,polariz_pre1_available,polariz_post1_available
0,6503,2256,2204,2162,2140,1911,1859


In [26]:
# Guardar dataset con merge DPI
out_dpi = paths["processed"] / "elections_master_dpi.parquet"
events_dpi.to_parquet(out_dpi, index=False)
out_dpi


PosixPath('/Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/elections_master_dpi.parquet')